<a href="https://colab.research.google.com/github/SayyadRizwan/GAN-based-Oral-Cancer-detection-with-GraphRag/blob/main/FINAL__OSCC_GraphRAG_with_GradCAM_ReportText_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OSCC GraphRAG Backend — Groq + Grad-CAM

Workflow: **EfficientNetB0 prediction → Grad-CAM heatmap → FAISS literature retrieval → Neo4j case record/context → Groq research report → FastAPI UI**.

This notebook extends the supplied working notebook. It loads the existing trained model (no retraining), creates a Grad-CAM overlay per uploaded image, passes a cautious Grad-CAM summary into retrieval and the report, stores Grad-CAM metadata on the Neo4j Case node, and displays the heatmap in the web UI.

Grad-CAM is a model explanation, not a lesion segmentation or clinical diagnosis.


In [1]:
!pip -q install fastapi uvicorn python-multipart pyngrok neo4j faiss-cpu sentence-transformers groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.7/390.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive, userdata
import os, json, uuid, time, threading, subprocess
from datetime import datetime, timezone

drive.mount("/content/drive")

def get_secret(name):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Missing Colab Secret: {name}")
    return value

NEO4J_URI = get_secret("NEO4J_URI")
NEO4J_USERNAME = get_secret("NEO4J_USERNAME")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE")

GROQ_API_KEY = get_secret("GROQ_API_KEY")
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")

MODEL_PATH = "/content/drive/MyDrive/Final Year Project/oral_cancer_efficientnetb0.keras"
FAISS_PATH = "/content/drive/MyDrive/OSCC_Project/graphrag_phase2/oscc_literature.faiss"
METADATA_PATH = "/content/drive/MyDrive/OSCC_Project/graphrag_phase2/chunk_metadata.json"

print("Secrets loaded.")

Mounted at /content/drive
Secrets loaded.


In [3]:
import tensorflow as tf
import numpy as np

# Load the already-trained model. This does not retrain or overwrite it.
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Existing EfficientNetB0 loaded:", MODEL_PATH)


Existing EfficientNetB0 loaded: /content/drive/MyDrive/Final Year Project/oral_cancer_efficientnetb0.keras


In [4]:
import faiss
from sentence_transformers import SentenceTransformer

faiss_index = faiss.read_index(FAISS_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    chunk_metadata = json.load(f)

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("FAISS index loaded:", faiss_index.ntotal, "vectors")
print("Metadata records:", len(chunk_metadata))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index loaded: 189 vectors
Metadata records: 189


In [5]:
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    session.run("RETURN 1 AS ok").single()

print("Neo4j connected.")

Neo4j connected.


In [6]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "openai/gpt-oss-20b"

test_response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "user", "content": "Reply with exactly: GROQ_CONNECTION_OK"}
    ]
)

print(test_response.choices[0].message.content.strip())

GROQ_CONNECTION_OK


In [7]:
from PIL import Image
import io
import numpy as np

IMAGE_SIZE = (224, 224)

def predict_image(image_bytes):
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    image = image.resize(IMAGE_SIZE)
    arr = np.array(image, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)

    pred = model.predict(arr, verbose=0)
    score = float(np.squeeze(pred))

    label = "OSCC" if score >= 0.5 else "Normal"
    confidence = score if label == "OSCC" else (1.0 - score)

    return label, confidence


def normalize_case(
    age=None, sex=None, tobacco=None, alcohol=None,
    lesion_site=None, magnification=None
):
    return {
        "age": age or "Not provided",
        "sex": sex or "Not provided",
        "tobacco": tobacco or "Not provided",
        "alcohol": alcohol or "Not provided",
        "lesion_site": lesion_site or "Not provided",
        "magnification": magnification or "Not provided",
    }


def get_oscc_graph_context():
    query = """
    MATCH (d:Disease {name:"Oral Squamous Cell Carcinoma"})
    OPTIONAL MATCH (d)-[r]-(x)
    RETURN d.name AS disease,
           type(r) AS relationship,
           labels(x) AS node_labels,
           x.name AS node_name,
           x.title AS node_title,
           x.paper_id AS paper_id,
           x.year AS year
    ORDER BY relationship, node_name, node_title
    """
    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        return [dict(r) for r in session.run(query)]


def build_case_query(prediction_label, confidence, patient_context):
    parts = [
        "oral squamous cell carcinoma histopathology",
        f"model prediction {prediction_label}",
        f"model confidence {confidence:.4f}",
    ]

    for key, value in patient_context.items():
        if value and value != "Not provided":
            parts.append(f"{key}: {value}")

    parts.append(
        "diagnosis histological features biomarkers research evidence"
    )
    return " ".join(parts)


def retrieve_literature(query, top_k=5):
    q = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = faiss_index.search(q, top_k)
    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(chunk_metadata):
            continue
        item = dict(chunk_metadata[idx])
        item["similarity"] = float(score)
        results.append(item)

    return results


def save_case_to_graph(
    case_id, filename, patient_context, prediction_label, confidence, gradcam_summary=None
):
    query = """
    MERGE (c:Case {case_id:$case_id})
    SET c.filename=$filename,
        c.age=$age,
        c.sex=$sex,
        c.tobacco=$tobacco,
        c.alcohol=$alcohol,
        c.lesion_site=$lesion_site,
        c.magnification=$magnification,
        c.prediction=$prediction,
        c.confidence=$confidence,
        c.gradcam_method=$gradcam_method,
        c.gradcam_target_layer=$gradcam_target_layer,
        c.gradcam_summary_json=$gradcam_summary_json,
        c.updated_at=$updated_at

    MERGE (p:Prediction {case_id:$case_id})
    SET p.label=$prediction,
        p.confidence=$confidence,
        p.updated_at=$updated_at

    MERGE (c)-[:HAS_MODEL_PREDICTION]->(p)

    WITH p
    OPTIONAL MATCH (d:Disease {
        name:"Oral Squamous Cell Carcinoma"
    })

    FOREACH (
        _ IN CASE
            WHEN $prediction = "OSCC" AND d IS NOT NULL
            THEN [1]
            ELSE []
        END |
        MERGE (p)-[:PREDICTS_CLASS]->(d)
    )
    """

    params = {
        "case_id": case_id,
        "filename": filename,
        **patient_context,
        "prediction": prediction_label,
        "confidence": float(confidence),
        "gradcam_method": (gradcam_summary or {}).get("method", "Not generated"),
        "gradcam_target_layer": (gradcam_summary or {}).get("target_layer", "Not generated"),
        "gradcam_summary_json": json.dumps(gradcam_summary or {}, ensure_ascii=False),
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }

    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        session.run(query, **params).consume()

print("Backend helper functions defined.")

Backend helper functions defined.


In [8]:
# Grad-CAM for the existing nested EfficientNetB0 model.
# The model architecture/weights remain unchanged; this code creates an explanation per image.
import base64
from PIL import Image, ImageOps
import io

IMAGE_SIZE = (224, 224)


def _find_efficientnet_backbone(loaded_model):
    for layer in loaded_model.layers:
        if isinstance(layer, tf.keras.Model) and "efficientnet" in layer.name.lower():
            return layer, loaded_model.layers.index(layer)
    raise ValueError(
        "Could not find a nested EfficientNet model. Check model.layers and update the backbone lookup."
    )


_BACKBONE, _BACKBONE_INDEX = _find_efficientnet_backbone(model)
try:
    _LAST_CONV = _BACKBONE.get_layer("top_conv")
except ValueError as exc:
    raise ValueError("EfficientNet backbone has no layer named 'top_conv'; inspect _BACKBONE.summary().") from exc

_GRADCAM_FEATURE_MODEL = tf.keras.Model(
    inputs=_BACKBONE.inputs,
    outputs=[_LAST_CONV.output, _BACKBONE.output],
    name="gradcam_feature_model"
)
_GRADCAM_HEAD_LAYERS = model.layers[_BACKBONE_INDEX + 1:]


def _predict_from_backbone_features(features):
    x = features
    for layer in _GRADCAM_HEAD_LAYERS:
        try:
            x = layer(x, training=False)
        except TypeError:
            x = layer(x)
    return x


def generate_gradcam(image_bytes, prediction_label=None):
    """Return Grad-CAM overlay as a data URL plus a cautious textual summary."""
    original = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    resized = original.resize(IMAGE_SIZE)
    image_array = np.asarray(resized, dtype=np.float32)
    input_tensor = tf.convert_to_tensor(image_array[None, ...], dtype=tf.float32)

    # Apply the same top-level preprocessing/augmentation layers that precede the
    # nested backbone during ordinary model inference.
    backbone_input = input_tensor
    for pre_layer in model.layers[:_BACKBONE_INDEX]:
        # Keras 3 InputLayer objects describe the input shape; they must not be
        # called with a tensor. Skip them, and apply actual preprocessing layers.
        if isinstance(pre_layer, tf.keras.layers.InputLayer):
            continue
        try:
            backbone_input = pre_layer(backbone_input, training=False)
        except TypeError as first_error:
            try:
                backbone_input = pre_layer(backbone_input)
            except TypeError:
                raise TypeError(
                    "Grad-CAM could not apply the pre-backbone layer "
                    f"'{pre_layer.name}' ({type(pre_layer).__name__}). "
                    "Inspect model.summary() and the top-level preprocessing layers."
                ) from first_error

    with tf.GradientTape() as tape:
        conv_features, backbone_features = _GRADCAM_FEATURE_MODEL(backbone_input, training=False)
        tape.watch(conv_features)
        output = _predict_from_backbone_features(backbone_features)
        score = tf.reshape(output, [-1])[0]
        # For a binary sigmoid model, use the signed target score for the predicted class.
        target_score = score if prediction_label == "OSCC" else -score

    gradients = tape.gradient(target_score, conv_features)
    if gradients is None:
        raise RuntimeError("Grad-CAM gradients are unavailable for the selected convolutional layer.")

    # Absolute gradient weighting also supports the Normal class in a sigmoid binary model.
    weights = tf.reduce_mean(tf.abs(gradients), axis=(1, 2), keepdims=True)
    heatmap = tf.reduce_sum(weights * tf.abs(conv_features), axis=-1)[0]
    heatmap = tf.maximum(heatmap, 0)
    max_value = tf.reduce_max(heatmap)
    heatmap = tf.where(max_value > 0, heatmap / (max_value + 1e-8), tf.zeros_like(heatmap))
    heatmap_np = heatmap.numpy()

    # Resize activation map and create a red/yellow overlay using PIL only.
    heat_img = Image.fromarray(np.uint8(np.clip(heatmap_np, 0, 1) * 255)).resize(original.size, Image.Resampling.BILINEAR)
    heat_arr = np.asarray(heat_img, dtype=np.float32) / 255.0
    # Simple heat palette: dark blue -> red -> yellow. The heatmap is for visual explanation, not segmentation.
    red = np.clip(1.5 * heat_arr, 0, 1)
    green = np.clip(1.5 * heat_arr - 0.35, 0, 1)
    blue = np.clip(0.65 - 1.5 * heat_arr, 0, 1)
    color_arr = np.stack([red, green, blue], axis=-1)
    color_img = Image.fromarray(np.uint8(color_arr * 255), mode="RGB")
    overlay = Image.blend(original, color_img, alpha=0.38)

    buffer = io.BytesIO()
    overlay.save(buffer, format="PNG")
    overlay_data_url = "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")

    # Compact, non-clinical description passed into GraphRAG and the report.
    high_activation_fraction = float(np.mean(heatmap_np >= 0.6))
    activation_fraction_pct = round(high_activation_fraction * 100, 1)
    summary = {
        "method": "Grad-CAM",
        "target_prediction": prediction_label or "Not specified",
        "target_layer": "EfficientNetB0/top_conv",
        # Keep dimensions available for technical debugging, but do not put them in report prose.
        "heatmap_size": [int(heatmap_np.shape[0]), int(heatmap_np.shape[1])],
        "normalized_activation_max": round(float(np.max(heatmap_np)), 4) if heatmap_np.size else 0.0,
        "fraction_of_map_with_activation_at_least_0_6": round(high_activation_fraction, 4),
        "report_text": (
            f"Grad-CAM was generated for the model's {prediction_label or 'unspecified'} prediction "
            f"using the EfficientNetB0 top_conv layer. Approximately {activation_fraction_pct}% "
            "of the normalized heatmap had activation values of 0.6 or higher. "
            "Highlighted regions indicate areas influencing the model output; they are not a verified lesion mask, "
            "histopathological finding, or clinical diagnosis."
        ),
        "interpretation_note": (
            "Highlighted areas indicate image regions influencing this model output; they are not a verified lesion mask or clinical finding."
        )
    }
    return overlay_data_url, summary


print("Grad-CAM initialized. Backbone:", _BACKBONE.name, "target layer: top_conv")


Grad-CAM initialized. Backbone: efficientnetb0 target layer: top_conv


In [9]:
REPORT_SYSTEM_INSTRUCTION = """
You are a research-support assistant for an oral cancer histopathology project.

Generate a careful, structured, research-oriented report from:
1. CNN prediction and confidence
2. Supplied patient/case metadata
3. Neo4j knowledge-graph context
4. Grad-CAM visual explanation summary (not a lesion mask)
5. Retrieved literature evidence

STRICT RULES:

1. Never invent patient information, clinical findings, or laboratory results.

2. Treat the CNN prediction strictly as a MODEL OUTPUT.
   Example:
   "The CNN classifier predicted the image as Normal with 89.83%
   model confidence."

3. Never convert CNN confidence into:
   - clinical probability
   - cancer probability
   - diagnostic certainty
   - probability that the patient is disease-free.

4. Do not claim that a histopathological feature is present or absent
   unless that feature is explicitly provided as verified evidence.

5. Do not claim that biomarkers such as Cyclin D1, EGFR, Ki-67,
   PD-L1, or p16 were detected in the submitted image unless
   biomarker-specific evidence is explicitly provided.

6. Knowledge-graph relationships provide RESEARCH CONTEXT.
   They do not establish that a risk factor, biomarker, or
   histological feature is present in this specific specimen.

7. Retrieved literature provides population-level or research evidence.
   Do not present literature findings as patient-specific observations.

8. Dataset filenames or labels such as "normal.jpg" may be reported
   as dataset metadata, but must not be presented as independent
   pathological confirmation.

9. Clearly separate:
   A. Case information
   B. CNN model result
   C. Knowledge-graph context
   D. Literature evidence
   E. Integrated interpretation
   F. Limitations

10. Do not provide a definitive clinical diagnosis.

11. Do not recommend treatment.

12. Use cautious research language such as:
   "the model predicts",
   "the knowledge graph associates",
   "the literature reports",
   "the retrieved evidence provides context",
   "this does not establish presence in the submitted image."

13. The final report must explicitly state that the output is
    intended for research/academic use and is not a clinical diagnosis.
"""


def generate_research_report(
    case_id, filename, patient_context, prediction_label, confidence,
    graph_context, retrieval_query, literature_evidence, gradcam_summary
):
    payload = {
        "case_id": case_id,
        "filename": filename,
        "patient_context": patient_context,
        "cnn_prediction": prediction_label,
        "cnn_confidence": confidence,
        "knowledge_graph_context": graph_context,
        # Send only human-readable Grad-CAM interpretation to the LLM report.
        # Technical image dimensions and raw heatmap metadata are excluded from report prose.
        "gradcam_summary": {
            "report_text": gradcam_summary.get("report_text", gradcam_summary.get("interpretation_note", "Grad-CAM summary unavailable.")),
            "method": gradcam_summary.get("method", "Grad-CAM"),
            "target_prediction": gradcam_summary.get("target_prediction", prediction_label),
            "target_layer": gradcam_summary.get("target_layer", "EfficientNetB0/top_conv"),
        },
        "retrieval_query": retrieval_query,
        "literature_evidence": literature_evidence,
    }

    prompt = f"""
Prepare a concise but detailed research report for this OSCC GraphRAG case.

Important formatting rules:
- In Case Summary, write the image filename exactly once. Do not include image dimensions, pixel sizes, image arrays, base64 data, or repeated metadata.
- The Grad-CAM result must be described in plain text in the report. Do not output the heatmap image, data URL, raw arrays, or technical image dimensions.
- In Grad-CAM Visual Explanation, use the supplied report_text and explain cautiously that highlighted regions influence the model output but are not confirmed lesions.
- Keep the report readable, with each case field appearing only once.

Use exactly these sections:

1. Case Summary
2. CNN Model Result
3. Grad-CAM Visual Explanation
4. Knowledge Graph Context
5. Literature Evidence
6. Integrated Research Interpretation
7. Limitations

Source data:

{json.dumps(payload, indent=2, ensure_ascii=False)}
"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": REPORT_SYSTEM_INSTRUCTION},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=2500
    )

    return response.choices[0].message.content.strip()


print("Groq report generator defined.")

Groq report generator defined.


In [10]:
from fastapi.responses import HTMLResponse
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(
    title="OSCC GraphRAG Backend",
    version="1.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "cnn": model is not None,
        "faiss": faiss_index is not None,
        "neo4j": neo4j_driver is not None,
        "groq": groq_client is not None,
        "groq_model": GROQ_MODEL,
        "model": "EfficientNetB0"
    }


@app.post("/analyze")
async def analyze(
    image: UploadFile = File(...),
    age: str = Form(""),
    sex: str = Form(""),
    tobacco: str = Form(""),
    alcohol: str = Form(""),
    lesion_site: str = Form(""),
    magnification: str = Form("")
):
    try:
        image_bytes = await image.read()
        if not image_bytes:
            raise HTTPException(status_code=400, detail="Uploaded image is empty.")

        prediction_label, confidence = predict_image(image_bytes)
        gradcam_data_url, gradcam_summary = generate_gradcam(image_bytes, prediction_label)

        patient_context = normalize_case(
            age=age, sex=sex, tobacco=tobacco, alcohol=alcohol,
            lesion_site=lesion_site, magnification=magnification
        )
        case_id = "CASE-" + uuid.uuid4().hex[:10].upper()
        graph_context = get_oscc_graph_context()

        # Grad-CAM metadata is added to the retrieval query so GraphRAG can retrieve
        # context relevant to the model output and its visual explanation.
        retrieval_query = build_case_query(prediction_label, confidence, patient_context)
        retrieval_query += (
            f" Grad-CAM visual explanation target {gradcam_summary['target_prediction']};"
            f" target layer {gradcam_summary['target_layer']};"
            " highlighted activation regions are model explanation, not confirmed lesion findings."
        )
        literature_evidence = retrieve_literature(retrieval_query, top_k=5)

        save_case_to_graph(
            case_id, image.filename or "uploaded_image", patient_context,
            prediction_label, confidence, gradcam_summary=gradcam_summary
        )

        report_text = generate_research_report(
            case_id=case_id,
            filename=image.filename or "uploaded_image",
            patient_context=patient_context,
            prediction_label=prediction_label,
            confidence=confidence,
            graph_context=graph_context,
            retrieval_query=retrieval_query,
            literature_evidence=literature_evidence,
            gradcam_summary=gradcam_summary
        )

        return {
            "case_id": case_id,
            "filename": image.filename or "uploaded_image",
            "patient_context": patient_context,
            "prediction": prediction_label,
            "confidence": float(confidence),
            "confidence_percent": round(float(confidence) * 100, 2),
            "gradcam_image": gradcam_data_url,
            "gradcam_summary": gradcam_summary,
            "graph_context": graph_context,
            "retrieval_query": retrieval_query,
            "literature_evidence": literature_evidence,
            "scope_note": (
                "Grad-CAM highlights model-influential regions; it is not a lesion mask or clinical finding. "
                "Literature and knowledge-graph evidence provide research context and do not establish "
                "that a specific feature is present in the uploaded image."
            ),
            "report": {
                "provider": "Groq",
                "model": GROQ_MODEL,
                "generated_at": datetime.now(timezone.utc).isoformat(),
                "text": report_text
            }
        }
    except HTTPException:
        raise
    except Exception as exc:
        import traceback
        error_trace = traceback.format_exc()
        print("\nFULL /analyze TRACEBACK:\n", error_trace)
        # Development diagnostic: include the failing Python line in the API response.
        # Remove the traceback from the response before public deployment.
        raise HTTPException(
            status_code=500,
            detail=(
                f"Analysis failed: {type(exc).__name__}: {str(exc)[:300]}\n"
                f"Traceback (last lines):\n{error_trace[-3500:]}"
            )
        )


print("FastAPI app created successfully.")
print("Routes: GET /, GET /health, POST /analyze, POST /chat, GET /docs")


# ------------------------------------------------------------
# Built-in frontend
# Polished single-page frontend served by FastAPI.
# No separate frontend project is required.

from pydantic import BaseModel, Field
from fastapi import HTTPException

class CaseChatRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=4000)
    case_context: dict = Field(default_factory=dict)
    report_text: str = Field(default="", max_length=20000)
    history: list[dict] = Field(default_factory=list)

@app.post("/chat")
def chat_with_groq(payload: CaseChatRequest):
    """Answer follow-up questions using the generated report and case context."""
    question = payload.question.strip()
    if not question:
        raise HTTPException(status_code=400, detail="Question cannot be empty.")

    system_prompt = (
        "You are an evidence-aware research assistant for an oral squamous cell "
        "carcinoma (OSCC) histopathology research prototype. Answer primarily from "
        "the supplied report and case context. Separate report statements, model "
        "outputs, retrieved literature, and cautious explanation. If the report "
        "does not support an answer, say so rather than inventing facts or citations. "
        "Do not diagnose or recommend treatment. Outputs require qualified clinician "
        "or pathologist review."
    )
    messages = [{"role": "system", "content": system_prompt}]
    for item in payload.history[-8:]:
        role, content = item.get("role"), item.get("content")
        if role in ("user", "assistant") and isinstance(content, str):
            messages.append({"role": role, "content": content[:4000]})

    case_context = json.dumps(payload.case_context, ensure_ascii=False, default=str)[:10000]
    report_text = payload.report_text[:20000]
    messages.append({
        "role": "user",
        "content": f"CASE CONTEXT:\n{case_context}\n\nGENERATED REPORT:\n{report_text}\n\nQUESTION:\n{question}"
    })

    try:
        result = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            temperature=0.2,
            max_tokens=1000
        )
        return {"answer": result.choices[0].message.content or "", "provider": "Groq", "model": GROQ_MODEL}
    except Exception as exc:
        raise HTTPException(status_code=502, detail=f"Groq request failed: {type(exc).__name__}: {exc}")



HTML_PAGE = '\\\n<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1.0">\n<title>OSCC GraphRAG | Research Console</title>\n<style>\n:root{--bg:#07111f;--panel:#0d1b2e;--panel2:#10243b;--line:#1e3652;--text:#eaf2ff;--muted:#8fa6bf;--accent:#36d1dc;--accent2:#6c7cff;--good:#35d07f;--warn:#ffbd59;--danger:#ff6b7a;--shadow:0 24px 70px rgba(0,0,0,.28)}\n*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:radial-gradient(circle at 15% 0%,#123254 0,transparent 34%),radial-gradient(circle at 90% 10%,#24235b 0,transparent 30%),var(--bg);color:var(--text);font-family:Inter,ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif}.wrap{max-width:1180px;margin:auto;padding:28px 20px 60px}.top{display:flex;justify-content:space-between;align-items:center;gap:20px;margin-bottom:24px}.brand{display:flex;align-items:center;gap:13px}.logo{width:46px;height:46px;border-radius:14px;background:linear-gradient(135deg,var(--accent),var(--accent2));display:grid;place-items:center;font-size:23px;font-weight:900;box-shadow:0 10px 30px rgba(54,209,220,.22)}.brand h1{font-size:18px;margin:0}.brand p{margin:3px 0 0;color:var(--muted);font-size:12px}.status-pill{border:1px solid #27506c;background:#0c2034;padding:9px 12px;border-radius:999px;font-size:12px;color:#b9d3e9}.dot{display:inline-block;width:7px;height:7px;background:var(--good);border-radius:50%;margin-right:7px;box-shadow:0 0 12px var(--good)}\n.hero{position:relative;overflow:hidden;padding:38px;border:1px solid var(--line);border-radius:26px;background:linear-gradient(135deg,rgba(16,36,59,.95),rgba(10,24,43,.82));box-shadow:var(--shadow);margin-bottom:22px}.hero:after{content:"";position:absolute;width:280px;height:280px;border-radius:50%;right:-90px;top:-120px;background:rgba(54,209,220,.12);filter:blur(8px)}.eyebrow{color:var(--accent);font-size:12px;font-weight:800;letter-spacing:1.5px;text-transform:uppercase}.hero h2{font-size:clamp(28px,5vw,48px);line-height:1.05;margin:10px 0 12px;max-width:760px}.hero p{color:var(--muted);max-width:760px;line-height:1.65;margin:0}.tech{display:flex;flex-wrap:wrap;gap:8px;margin-top:20px}.tech span{padding:7px 10px;border:1px solid var(--line);border-radius:999px;background:#0a1829;color:#b8cae0;font-size:11px}\n.layout{display:grid;grid-template-columns:1fr 360px;gap:20px}.card{background:rgba(13,27,46,.94);border:1px solid var(--line);border-radius:22px;box-shadow:var(--shadow);padding:22px}.card h3{margin:0 0 5px;font-size:18px}.sub{color:var(--muted);font-size:12px;margin:0 0 18px}.drop{border:1.5px dashed #376080;border-radius:18px;padding:30px 18px;text-align:center;background:linear-gradient(180deg,#0c1c2f,#0a1727);cursor:pointer;transition:.2s}.drop:hover,.drop.drag{border-color:var(--accent);background:#0d2238}.upload-icon{font-size:34px;margin-bottom:10px}.drop strong{display:block;font-size:15px}.drop small{display:block;color:var(--muted);margin-top:7px}.drop input{display:none}.preview{display:none;margin-top:14px;grid-template-columns:130px 1fr;gap:14px;padding:12px;border:1px solid var(--line);border-radius:15px;background:#091728}.preview img{width:130px;height:110px;object-fit:cover;border-radius:10px}.filemeta{display:flex;flex-direction:column;justify-content:center;gap:5px}.filemeta b{font-size:13px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.filemeta span{font-size:11px;color:var(--muted)}.grid{display:grid;grid-template-columns:repeat(2,1fr);gap:13px;margin-top:16px}.field label{display:block;font-size:11px;color:#9db2c8;margin:0 0 6px;font-weight:700}.field input,.field select{width:100%;padding:12px 13px;border:1px solid var(--line);border-radius:11px;background:#091728;color:var(--text);outline:none}.field input:focus,.field select:focus{border-color:var(--accent);box-shadow:0 0 0 3px rgba(54,209,220,.08)}.full{grid-column:1/-1}.actions{display:flex;gap:10px;margin-top:16px}.primary,.secondary{border:0;border-radius:12px;padding:13px 16px;font-weight:800;cursor:pointer}.primary{flex:1;color:#04131d;background:linear-gradient(135deg,var(--accent),#7ce7df);box-shadow:0 12px 25px rgba(54,209,220,.14)}.primary:disabled{opacity:.55;cursor:wait}.secondary{background:#10243b;color:#bcd0e4;border:1px solid var(--line)}.notice{margin-top:13px;padding:11px 13px;border-radius:11px;background:#0b1b2d;border:1px solid var(--line);color:#91a8bf;font-size:11px;line-height:1.5}.side-stat{display:grid;grid-template-columns:1fr 1fr;gap:10px}.stat{padding:14px;border:1px solid var(--line);border-radius:15px;background:#0a1829}.stat b{display:block;font-size:17px;margin-top:4px}.stat span{font-size:10px;color:var(--muted)}.steps{display:flex;flex-direction:column;gap:11px;margin-top:18px}.step{display:flex;gap:11px;align-items:flex-start;color:#8ea4bb;font-size:12px}.step .n{width:25px;height:25px;border-radius:8px;border:1px solid var(--line);display:grid;place-items:center;flex:none}.step.active{color:#d9e8f8}.step.active .n{border-color:var(--accent);color:var(--accent)}\n#status{min-height:20px;margin-top:13px;font-size:12px;color:var(--muted)}.loader{display:none;height:5px;background:#10243b;border-radius:999px;overflow:hidden;margin-top:8px}.loader i{display:block;width:35%;height:100%;background:linear-gradient(90deg,var(--accent),var(--accent2));animation:load 1.1s infinite}.results{display:none;margin-top:20px}.result-head{display:flex;justify-content:space-between;gap:15px;align-items:center;flex-wrap:wrap}.case{font-size:11px;color:var(--muted)}.badges{display:flex;flex-wrap:wrap;gap:8px}.badge{padding:8px 11px;border-radius:999px;border:1px solid var(--line);background:#0a1829;font-size:11px;color:#c6d7e8}.badge strong{color:white}.result-grid{display:grid;grid-template-columns:270px 1fr;gap:18px;margin-top:18px}.confidence{display:grid;place-items:center;min-height:245px;border:1px solid var(--line);border-radius:18px;background:#091728}.ring{--p:0deg;width:150px;height:150px;border-radius:50%;display:grid;place-items:center;background:conic-gradient(var(--accent) var(--p),#19314a 0deg);position:relative}.ring:after{content:"";position:absolute;width:116px;height:116px;background:#091728;border-radius:50%}.ring div{position:relative;z-index:1;text-align:center}.ring b{display:block;font-size:25px}.ring span{font-size:10px;color:var(--muted)}.prediction{margin-top:12px;font-size:13px;font-weight:800}.tabs{display:flex;gap:7px;border-bottom:1px solid var(--line);padding-bottom:8px}.tab{border:0;background:transparent;color:#8198b0;padding:8px 10px;border-radius:8px;cursor:pointer;font-size:11px;font-weight:700}.tab.active{background:#102b43;color:#cfe9f3}.panel{display:none;padding-top:14px}.panel.active{display:block}.report{white-space:pre-wrap;line-height:1.7;color:#c8d7e7;font-size:13px}.evidence{display:grid;gap:9px}.ev{border:1px solid var(--line);background:#091728;border-radius:12px;padding:12px}.ev b{font-size:12px}.ev p{margin:5px 0 0;color:#8fa7bd;font-size:11px;line-height:1.5}.jsonbox{white-space:pre-wrap;background:#07111f;border:1px solid var(--line);padding:13px;border-radius:12px;color:#9db5cb;font-size:11px;line-height:1.55;max-height:360px;overflow:auto}.footer{margin-top:24px;color:#617991;text-align:center;font-size:10px}.toast{position:fixed;right:18px;bottom:18px;background:#102b43;border:1px solid #31536e;color:#d9ebf7;padding:12px 15px;border-radius:12px;box-shadow:var(--shadow);display:none;font-size:12px;z-index:10}@keyframes load{0%{transform:translateX(-120%)}100%{transform:translateX(330%)}}@media(max-width:900px){.layout,.result-grid{grid-template-columns:1fr}.side{order:-1}.side-stat{grid-template-columns:repeat(4,1fr)}}@media(max-width:620px){.wrap{padding:16px 12px 40px}.hero{padding:25px}.grid{grid-template-columns:1fr}.full{grid-column:auto}.top{align-items:flex-start}.status-pill{display:none}.side-stat{grid-template-columns:1fr 1fr}.preview{grid-template-columns:90px 1fr}.preview img{width:90px;height:85px}.actions{flex-direction:column}}\n</style>\n</head>\n<body>\n<div class="wrap">\n  <header class="top"><div class="brand"><div class="logo">◈</div><div><h1>OSCC GraphRAG</h1><p>Research &amp; knowledge intelligence console</p></div></div><div class="status-pill"><span class="dot"></span>Backend ready</div></header>\n  <section class="hero"><div class="eyebrow">AI-assisted histopathology research</div><h2>Explore an OSCC case with vision + knowledge.</h2><p>Upload a histopathology image and combine EfficientNetB0 prediction with retrieved literature, a Neo4j knowledge graph and a Groq-generated research report.</p><div class="tech"><span>EfficientNetB0</span><span>FAISS Retrieval</span><span>Neo4j Graph</span><span>Groq LLM</span></div></section>\n  <div class="layout">\n    <main>\n      <section class="card"><h3>New analysis</h3><p class="sub">Provide the image and optional case context.</p>\n        <form id="analysisForm">\n          <label class="drop" id="dropZone"><input id="image" name="image" type="file" accept="image/*" required><div class="upload-icon">⇧</div><strong>Drop histopathology image here</strong><small>or click to browse · PNG, JPG, JPEG</small></label>\n          <div class="preview" id="preview"><img id="previewImg"><div class="filemeta"><b id="fileName">—</b><span id="fileSize">—</span><span>Image selected and ready for analysis</span></div></div>\n          <div class="grid">\n            <div class="field"><label>AGE</label><input id="age" name="age" placeholder="e.g. 55"></div>\n            <div class="field"><label>SEX</label><select id="sex" name="sex"><option value="">Select</option><option>Male</option><option>Female</option><option>Other</option></select></div>\n            <div class="field"><label>TOBACCO</label><input id="tobacco" name="tobacco" placeholder="Yes / No"></div>\n            <div class="field"><label>ALCOHOL</label><input id="alcohol" name="alcohol" placeholder="Yes / No"></div>\n            <div class="field"><label>LESION SITE</label><input id="lesion_site" name="lesion_site" placeholder="e.g. Buccal mucosa"></div>\n            <div class="field"><label>MAGNIFICATION</label><input id="magnification" name="magnification" placeholder="e.g. 40x"></div>\n          </div>\n          <div class="actions"><button class="primary" id="submitBtn" type="submit">Analyze case →</button><button class="secondary" id="resetBtn" type="button">Reset</button></div>\n          <div class="loader" id="loader"><i></i></div><div id="status"></div>\n          <div class="notice">Research support only. Model confidence is not a clinical probability, and the retrieved literature does not prove that a specific histological feature is present in the uploaded image.</div>\n        </form>\n      </section>\n      <section class="card results" id="resultCard">\n        <div class="result-head"><div><h3>Case analysis</h3><div class="case" id="caseLine">—</div></div><div class="badges" id="badges"></div></div>\n        <div class="result-grid"><div class="confidence"><div><div class="ring" id="ring"><div><b id="confidenceValue">0%</b><span>model confidence</span></div></div><div class="prediction" id="prediction">Prediction: —</div></div></div>\n          <div><div class="tabs"><button class="tab active" data-tab="reportPanel">Research report</button><button class="tab" data-tab="litPanel">Literature</button><button class="tab" data-tab="graphPanel">Knowledge graph</button></div>\n            <div class="panel active" id="reportPanel"><div class="report" id="report"></div><button class="secondary" id="downloadBtn" style="margin-top:12px">Download report</button></div>\n            <div class="panel" id="litPanel"><div class="evidence" id="literature"></div></div>\n            <div class="panel" id="graphPanel"><div class="jsonbox" id="graph"></div></div>\n          </div>\n        </div>\n        <section class="card" id="gradcamCard" style="margin-top:18px;background:#091728">\n          <h3>Grad-CAM visual explanation</h3>\n          <p class="sub">Highlighted regions show image areas influencing the model output. They are not a verified lesion segmentation.</p>\n          <img id="gradcamImage" alt="Grad-CAM overlay" style="display:block;width:100%;max-width:560px;max-height:560px;object-fit:contain;border-radius:12px;border:1px solid var(--line);background:#07111f">\n          <div class="jsonbox" id="gradcamSummary" style="margin-top:12px"></div>\n        </section>\n      </section>\n    </main>\n    <aside class="side"><section class="card"><h3>Pipeline</h3><p class="sub">What happens after you submit</p><div class="side-stat"><div class="stat"><span>VISION</span><b>CNN</b></div><div class="stat"><span>SEARCH</span><b>FAISS</b></div><div class="stat"><span>GRAPH</span><b>Neo4j</b></div><div class="stat"><span>REPORT</span><b>Groq</b></div></div><div class="steps"><div class="step active"><div class="n">1</div><div><b>Image inference</b><br>EfficientNetB0 produces the model output.</div></div><div class="step active"><div class="n">2</div><div><b>Evidence retrieval</b><br>Relevant research chunks are retrieved from FAISS.</div></div><div class="step active"><div class="n">3</div><div><b>Graph context</b><br>Neo4j contributes structured case knowledge.</div></div><div class="step active"><div class="n">4</div><div><b>Research synthesis</b><br>Groq turns the supplied evidence into a report.</div></div></div></section></aside>\n  </div>\n  <div class="footer">OSCC GraphRAG · Academic/research prototype · Not a diagnostic device</div>\n</div><div class="toast" id="toast"></div>\n<script>\nconst $=id=>document.getElementById(id); const form=$(\'analysisForm\'), file=$(\'image\'), drop=$(\'dropZone\'), preview=$(\'preview\'); let latest=null;\nfunction toast(msg){const t=$(\'toast\');t.textContent=msg;t.style.display=\'block\';setTimeout(()=>t.style.display=\'none\',2600)}\nfunction formatBytes(n){if(!n)return \'0 B\';const u=[\'B\',\'KB\',\'MB\',\'GB\'];let i=0;while(n>=1024&&i<3){n/=1024;i++}return n.toFixed(i?1:0)+\' \'+u[i]}\nfunction showFile(f){if(!f)return; $(\'fileName\').textContent=f.name;$(\'fileSize\').textContent=formatBytes(f.size);$(\'previewImg\').src=URL.createObjectURL(f);preview.style.display=\'grid\'}\nfile.addEventListener(\'change\',()=>showFile(file.files[0]));[\'dragenter\',\'dragover\'].forEach(e=>drop.addEventListener(e,x=>{x.preventDefault();drop.classList.add(\'drag\')}));[\'dragleave\',\'drop\'].forEach(e=>drop.addEventListener(e,x=>{x.preventDefault();drop.classList.remove(\'drag\')}));drop.addEventListener(\'drop\',e=>{const f=e.dataTransfer.files[0];if(f&&f.type.startsWith(\'image/\')){file.files=e.dataTransfer.files;showFile(f)}else toast(\'Please drop an image file.\')});\n$(\'resetBtn\').onclick=()=>{form.reset();preview.style.display=\'none\';$(\'resultCard\').style.display=\'none\';$(\'status\').textContent=\'\';latest=null};\ndocument.querySelectorAll(\'.tab\').forEach(b=>b.onclick=()=>{document.querySelectorAll(\'.tab\').forEach(x=>x.classList.remove(\'active\'));document.querySelectorAll(\'.panel\').forEach(x=>x.classList.remove(\'active\'));b.classList.add(\'active\');$(b.dataset.tab).classList.add(\'active\')});\nfunction esc(v){return String(v??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]))}\nfunction literatureHTML(items){if(!Array.isArray(items)||!items.length)return \'<div class="ev"><b>No literature evidence returned</b><p>The retrieval step returned no displayable records.</p></div>\';return items.map((x,i)=>{if(typeof x===\'string\')return `<div class="ev"><b>Evidence ${i+1}</b><p>${esc(x)}</p></div>`;const title=x.title||x.paper_title||x.name||`Evidence ${i+1}`;const body=x.text||x.content||x.chunk||x.snippet||x.abstract||JSON.stringify(x);return `<div class="ev"><b>${esc(title)}</b><p>${esc(body)}</p></div>`}).join(\'\')}\nform.addEventListener(\'submit\',async e=>{e.preventDefault();if(!file.files[0])return toast(\'Select a histopathology image first.\');const fd=new FormData();fd.append(\'image\',file.files[0]);[\'age\',\'sex\',\'tobacco\',\'alcohol\',\'lesion_site\',\'magnification\'].forEach(id=>fd.append(id,$(id).value));$(\'submitBtn\').disabled=true;$(\'loader\').style.display=\'block\';$(\'status\').textContent=\'Running image inference → retrieval → graph → report…\';$(\'resultCard\').style.display=\'none\';try{const r=await fetch(\'/analyze\',{method:\'POST\',body:fd});let d;try{d=await r.json()}catch(_){const body=await r.text().catch(()=>\'\');throw new Error(\'Backend returned non-JSON (HTTP \'+r.status+\'): \'+body.slice(0,300))}if(!r.ok)throw new Error(d.detail|| (\'Analysis failed (HTTP \'+r.status+\').\'));latest=d;window.latest=d;const pct=Number(d.confidence_percent??(Number(d.confidence||0)*100));$(\'confidenceValue\').textContent=pct.toFixed(1)+\'%\';$(\'ring\').style.setProperty(\'--p\',Math.max(0,Math.min(100,pct))*3.6+\'deg\');$(\'prediction\').textContent=\'Prediction: \'+(d.prediction||\'N/A\');$(\'caseLine\').textContent=(d.case_id||\'CASE\')+\' · \'+(d.filename||\'image\');$(\'badges\').innerHTML=`<span class="badge">Model <strong>EfficientNetB0</strong></span><span class="badge">LLM <strong>Groq</strong></span>`;$(\'report\').textContent=d.report?.text||\'No report returned.\';if(d.gradcam_image){$(\'gradcamImage\').src=d.gradcam_image;$(\'gradcamImage\').style.display=\'block\'}else{$(\'gradcamImage\').removeAttribute(\'src\');$(\'gradcamImage\').style.display=\'none\'}$(\'gradcamSummary\').textContent=JSON.stringify(d.gradcam_summary||{},null,2);$(\'literature\').innerHTML=literatureHTML(d.literature_evidence);$(\'graph\').textContent=JSON.stringify(d.graph_context??{},null,2);$(\'resultCard\').style.display=\'block\';$(\'status\').textContent=\'Analysis completed successfully.\';toast(\'Case analysis complete.\');setTimeout(()=>$(\'resultCard\').scrollIntoView({behavior:\'smooth\',block:\'start\'}),80)}catch(err){$(\'status\').textContent=\'Error: \'+err.message;toast(\'Analysis failed.\')}finally{$(\'submitBtn\').disabled=false;$(\'loader\').style.display=\'none\'}});\n$(\'downloadBtn\').onclick=()=>{if(!latest)return;const text=`OSCC GraphRAG Research Report\\n\\nCase: ${latest.case_id||\'\'}\\nPrediction: ${latest.prediction||\'\'}\\nConfidence: ${latest.confidence_percent??\'\'}%\\n\\n${latest.report?.text||\'\'}\\n\\nScope note:\\n${latest.scope_note||\'\'}`;const a=document.createElement(\'a\');a.href=URL.createObjectURL(new Blob([text],{type:\'text/plain\'}));a.download=(latest.case_id||\'OSCC_case\')+\'_report.txt\';a.click();URL.revokeObjectURL(a.href)};\n</script></body></html>\n'

# OSCC_REPORT_CHAT_UI_INJECTION: enhance the existing UI after HTML_PAGE is fully defined.
if "reportChatCard" not in HTML_PAGE:
    _chat_markup = r"""
<section class="card" id="reportChatCard" style="margin-top:22px">
  <div class="section-head">
    <div>
      <div class="eyebrow">REPORT ASSISTANT</div>
      <h2>Ask questions about this report</h2>
      <p class="muted">Answers are grounded in the generated report and case context. Research support only.</p>
    </div>
    <span class="status-pill">Groq AI</span>
  </div>
  <div id="reportChatMessages" class="chat-messages" aria-live="polite">
    <div class="chat-empty">Generate a report first, then ask about its findings, confidence, evidence, limitations, or terminology.</div>
  </div>
  <div class="chat-compose">
    <textarea id="reportChatQuestion" rows="2" placeholder="Ask a question about the generated report..." aria-label="Question about report"></textarea>
    <button id="reportChatSend" type="button">Send question</button>
  </div>
  <div class="chat-suggestions">
    <button type="button" class="chat-suggestion" data-question="Summarize the key findings in this report.">Summarize findings</button>
    <button type="button" class="chat-suggestion" data-question="Explain the model prediction and confidence in plain language.">Explain prediction</button>
    <button type="button" class="chat-suggestion" data-question="What are the limitations and uncertainties in this report?">Limitations</button>
    <button type="button" class="chat-suggestion" data-question="Explain the literature evidence and how it relates to this case.">Literature evidence</button>
  </div>
</section>
"""
    _chat_css = r"""
.chat-messages{display:flex;flex-direction:column;gap:12px;max-height:340px;overflow:auto;padding:14px;border:1px solid var(--line,#dbe4e8);border-radius:14px;background:rgba(5,15,28,.45);margin:14px 0}
.chat-empty{color:var(--muted,#8fa6bf);font-size:13px;padding:8px;line-height:1.6}
.chat-message{max-width:92%;padding:12px 14px;border-radius:14px;line-height:1.6;white-space:pre-wrap;overflow-wrap:anywhere;font-size:13px}
.user-message{align-self:flex-end;background:#153c4b;color:#e7fbff}
.assistant-message{align-self:flex-start;background:#0a1727;border:1px solid var(--line,#1e3652);color:var(--text,#eaf2ff)}
.chat-role{font-size:10px;text-transform:uppercase;letter-spacing:.08em;font-weight:800;opacity:.7;margin-bottom:5px}
.chat-compose{display:flex;gap:10px;align-items:stretch}
.chat-compose textarea{flex:1;min-width:0;resize:vertical;border:1px solid var(--line,#1e3652);border-radius:12px;padding:12px;background:#091728;color:var(--text,#eaf2ff);font:inherit}
.chat-suggestions{display:flex;flex-wrap:wrap;gap:8px;margin-top:10px}
.chat-suggestion{border:1px solid var(--line,#1e3652);border-radius:999px;padding:7px 11px;background:#10243b;color:#c6d7e8;cursor:pointer;font-size:11px}
#reportChatCard .status-pill{display:inline-block;border-radius:999px;padding:6px 10px;background:#0c302d;color:#8ce8d4;font-size:11px;font-weight:800}
@media(max-width:640px){.chat-compose{flex-direction:column}.chat-message{max-width:100%}}
"""
    _chat_js = r"""
<script>
(function(){
  const $ = id => document.getElementById(id);
  const card = $('reportChatCard');
  if (!card) return;
  let history = [];
  function addMessage(role, content) {
    const wrap = document.createElement('div');
    wrap.className = 'chat-message ' + (role === 'user' ? 'user-message' : 'assistant-message');
    const label = document.createElement('div');
    label.className = 'chat-role';
    label.textContent = role === 'user' ? 'You' : 'Report Assistant';
    const body = document.createElement('div');
    body.className = 'chat-body';
    body.textContent = content;
    wrap.append(label, body);
    $('reportChatMessages').appendChild(wrap);
    $('reportChatMessages').scrollTop = $('reportChatMessages').scrollHeight;
  }
  function latestReport() {
    return window.latest || window.latestResult || window.lastResult ||
           window.analysisResult || window.currentReport || null;
  }
  async function ask(question) {
    const q = (question || $('reportChatQuestion').value || '').trim();
    if (!q) return;
    const result = latestReport();
    if (!result) {
      addMessage('assistant', 'Please analyze an image and generate a report first.');
      return;
    }
    const reportText = (result.report && result.report.text) ||
                       result.report_text || result.reportText || '';
    const context = {
      case_id: result.case_id || '',
      prediction: result.prediction || '',
      confidence: result.confidence_percent ?? result.confidence ?? null,
      patient_context: result.patient_context || {},
      graph_context: result.graph_context || '',
      gradcam_summary: result.gradcam_summary || {},
      literature_evidence: result.literature_evidence || '',
      scope_note: result.scope_note || ''
    };
    addMessage('user', q);
    $('reportChatQuestion').value = '';
    $('reportChatSend').disabled = true;
    $('reportChatSend').textContent = 'Thinking…';
    try {
      const response = await fetch('/chat', {
        method: 'POST',
        headers: {'Content-Type':'application/json'},
        body: JSON.stringify({question:q, case_context:context, report_text:reportText, history})
      });
      const data = await response.json();
      if (!response.ok) throw new Error(data.detail || 'Chat request failed.');
      const answer = data.answer || 'No answer returned.';
      addMessage('assistant', answer);
      history.push({role:'user',content:q},{role:'assistant',content:answer});
      history = history.slice(-8);
    } catch (error) {
      addMessage('assistant', 'Could not answer: ' + error.message + '. Check backend and Groq configuration.');
    } finally {
      $('reportChatSend').disabled = false;
      $('reportChatSend').textContent = 'Send question';
    }
  }
  $('reportChatSend').addEventListener('click', () => ask());
  $('reportChatQuestion').addEventListener('keydown', e => {
    if (e.key === 'Enter' && !e.shiftKey) { e.preventDefault(); ask(); }
  });
  card.querySelectorAll('[data-question]').forEach(btn => btn.addEventListener('click', () => ask(btn.dataset.question)));
})();
</script>
"""
    # Insert the chat into the existing page near the end, before the body closes.
    if "</body>" in HTML_PAGE:
        HTML_PAGE = HTML_PAGE.replace("</body>", _chat_markup + _chat_js + "</body>", 1)
    else:
        HTML_PAGE += _chat_markup + _chat_js
    if "</style>" in HTML_PAGE:
        HTML_PAGE = HTML_PAGE.replace("</style>", _chat_css + "</style>", 1)
    else:
        HTML_PAGE = "<style>" + _chat_css + "</style>" + HTML_PAGE


@app.get("/", response_class=HTMLResponse)
def home():
    return HTML_PAGE



print("Enhanced frontend route created successfully.")
print("Website URL: PUBLIC_BACKEND_URL + '/'")


FastAPI app created successfully.
Routes: GET /, GET /health, POST /analyze, POST /chat, GET /docs
Enhanced frontend route created successfully.
Website URL: PUBLIC_BACKEND_URL + '/'


## Start FastAPI and create a fresh temporary ngrok tunnel

Run after all setup, model, Grad-CAM, FAISS, Neo4j, Groq, and helper cells complete. The backend returns JSON error details for analysis failures so the frontend can show a useful error message.


In [11]:
import requests, time, threading
from pyngrok import ngrok
import uvicorn

if not globals().get("NGROK_AUTHTOKEN"):
    raise RuntimeError("NGROK_AUTHTOKEN is missing. Add it to Colab Secrets and rerun the setup cell.")
ngrok.set_auth_token(NGROK_AUTHTOKEN)

def _local_health_ok():
    try:
        return requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200
    except requests.RequestException:
        return False

try:
    ngrok.kill()
except Exception as exc:
    print("ngrok cleanup note:", exc)

if not _local_health_ok():
    _server_thread = threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"),
        daemon=True
    )
    _server_thread.start()
    for _ in range(60):
        if _local_health_ok():
            break
        time.sleep(1)

if not _local_health_ok():
    raise RuntimeError("FastAPI is not responding on port 8000. Check Uvicorn logs and prior initialization cells.")

_tunnel = ngrok.connect(addr=8000, proto="http")
PUBLIC_BACKEND_URL = _tunnel.public_url.rstrip("/")

print("Website:", PUBLIC_BACKEND_URL + "/")
print("Health:", PUBLIC_BACKEND_URL + "/health")
print("API docs:", PUBLIC_BACKEND_URL + "/docs")

INFO:     Started server process [1721]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:36322 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:36332 - "GET /health HTTP/1.1" 200 OK
Website: https://trustless-angular-revisit.ngrok-free.dev/
Health: https://trustless-angular-revisit.ngrok-free.dev/health
API docs: https://trustless-angular-revisit.ngrok-free.dev/docs


## Verify public backend health

In [12]:
import requests, json
if not globals().get("PUBLIC_BACKEND_URL"):
    raise RuntimeError("Run the startup/tunnel cell first.")
r = requests.get(PUBLIC_BACKEND_URL + "/health", timeout=30)
print("HTTP status:", r.status_code)
try:
    print(json.dumps(r.json(), indent=2))
except ValueError:
    print(r.text[:2000])

INFO:     34.124.250.204:0 - "GET /health HTTP/1.1" 200 OK
HTTP status: 200
{
  "status": "ok",
  "cnn": true,
  "faiss": true,
  "neo4j": true,
  "groq": true,
  "groq_model": "openai/gpt-oss-20b",
  "model": "EfficientNetB0"
}


## Verify routes

In [13]:
for route in app.routes:
    print(sorted(getattr(route, "methods", []) or []), route.path)

['GET', 'HEAD'] /openapi.json
['GET', 'HEAD'] /docs
['GET', 'HEAD'] /docs/oauth2-redirect
['GET', 'HEAD'] /redoc
['GET'] /health
['POST'] /analyze
['POST'] /chat
['GET'] /
